In [ ]:
import cv2  # Qué: importa OpenCV. Cómo: expone el submódulo `legacy` con los trackers clásicos (CSRT, MOSSE) y las utilidades de captura/GUI. Por qué: es la única dependencia necesaria para el seguimiento de objetos en video.

In [ ]:
#tracker = cv2.legacy.TrackerMOSSE_create()  # Qué: (comentado) crearía un tracker MOSSE. Cómo: MOSSE usa filtros de correlación adaptativos, muy livianos computacionalmente. Por qué se deja comentado: se usa como referencia/alternativa más rápida pero menos precisa que CSRT; se retoma en la práctica.

tracker = cv2.legacy.TrackerCSRT_create()  # Qué: crea un tracker CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability). Cómo: usa filtros de correlación discriminativos considerando fiabilidad espacial y por canal para reajustar el cuadro delimitador en cada frame. Por qué: se elige CSRT porque es más preciso que MOSSE ante cambios de escala/oclusión parcial, a costa de ser más lento (trade-off precisión vs. velocidad).

In [ ]:
cap = cv2.VideoCapture(0)  # Qué: abre la webcam por defecto (índice 0). Por qué: fuente de video sobre la que se va a seguir el objeto elegido.

ret, frame = cap.read()  # Qué: captura un único cuadro inicial. Cómo: `ret` indica éxito y `frame` es la imagen BGR. Por qué: se necesita un primer cuadro estático para que el usuario seleccione manualmente el objeto a rastrear, antes de entrar al loop continuo.

bbox = cv2.selectROI("Selecciona el objeto", frame, False)  # Qué: abre una ventana interactiva donde el usuario dibuja con el mouse un rectángulo (ROI, Region Of Interest) sobre el objeto a seguir. Cómo: devuelve la tupla (x, y, w, h) del rectángulo seleccionado; el tercer argumento `False` desactiva el "crosshair" centrado en la selección. Por qué: el tracker necesita una caja delimitadora inicial para aprender la apariencia del objeto que va a seguir.

cv2.destroyWindow("Selecciona el objeto")  # Qué: cierra específicamente la ventana de selección. Por qué: ya cumplió su propósito (obtener bbox) y no debe seguir ocupando pantalla durante el seguimiento.


# Inicializar el rastreador con el cuadro y la caja delimitadora seleccionada

In [ ]:
ok = tracker.init(frame, bbox)  # Qué: inicializa el tracker con el cuadro inicial y la caja delimitadora seleccionada. Cómo: CSRT extrae características de la región `bbox` dentro de `frame` para construir su modelo de apariencia inicial. Por qué: es el paso obligatorio antes de poder llamar a `tracker.update()` en cada cuadro nuevo.

In [ ]:
while True:  # Qué: procesa la webcam en vivo, cuadro por cuadro, actualizando el seguimiento hasta que el usuario salga.
    ret, frame = cap.read()  # Qué: captura el siguiente cuadro del video. Cómo: `ret` indica éxito, `frame` es la imagen BGR actual.

    # Actualizar el rastreador con el cuadro actual
    success, bbox = tracker.update(frame)  # Qué: recalcula la posición del objeto en el nuevo cuadro. Cómo: CSRT busca, alrededor de la última posición conocida, la región que mejor coincide con su modelo de apariencia y devuelve `success` (si logró seguirlo) y la nueva `bbox` (x, y, w, h). Por qué: es el mecanismo central del seguimiento: evita tener que re-detectar el objeto desde cero en cada frame.

    # Si el rastreo es exitoso, dibujar el cuadro delimitador
    if success:  # Qué: rama que se ejecuta cuando el tracker sigue confiando en su seguimiento.
        x, y, w, h = map(int, bbox)  # Qué: descompone la tupla bbox en sus 4 componentes y los convierte a enteros. Por qué: las funciones de dibujo de OpenCV (rectangle, putText) requieren coordenadas enteras en píxeles, y `bbox` puede venir con valores flotantes.
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)  # Qué: dibuja un rectángulo azul (BGR: 255,0,0) alrededor del objeto rastreado. Cómo: traza desde la esquina superior izquierda (x,y) hasta la inferior derecha (x+w, y+h) con grosor de línea 2. Por qué: da feedback visual inmediato de dónde cree el tracker que está el objeto.
        cv2.putText(frame, "Seguimiento", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)  # Qué: escribe la etiqueta "Seguimiento" arriba del rectángulo. Cómo: usa la fuente FONT_HERSHEY_SIMPLEX con escala 0.6 y el mismo color azul, posicionada 10px arriba de `y` para no tapar el rectángulo. Por qué: refuerza visualmente el estado "objeto encontrado".
    else:  # Qué: rama que se ejecuta cuando el tracker perdió el objeto (p. ej. por oclusión total o que salió de cuadro).
        cv2.putText(frame, "Perdido", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)  # Qué: muestra el texto "Perdido" en rojo en una posición fija de la pantalla. Por qué: informa al usuario que debe volver a seleccionar el objeto o que el seguimiento falló, en vez de dibujar una bbox inválida.

    cv2.imshow("Seguimiento de Objeto", frame)  # Qué: muestra el cuadro actual con las anotaciones dibujadas. Por qué: es la salida visual en vivo del pipeline de seguimiento.

    if cv2.waitKey(30) == ord('q'):  # Qué: espera 30 ms por una tecla y compara con 'q'. Cómo: el delay de 30ms (a diferencia del 1ms de otros notebooks) regula aproximadamente la tasa de refresco a ~33 FPS. Por qué: da tiempo a que el tracker CSRT (más pesado computacionalmente) procese cada cuadro sin saturar el loop.
        break  # Qué: corta el bucle infinito. Por qué: única salida controlada del loop de seguimiento.
    
cap.release()  # Qué: libera el dispositivo de cámara. Por qué: evita bloquear la webcam para otros procesos.
cv2.destroyAllWindows()  # Qué: cierra todas las ventanas abiertas de OpenCV. Por qué: limpieza de recursos de GUI.


## 🧪 Práctica
Reforzá lo aprendido en este módulo resolviendo los ejercicios guiados en [`practicas/7_practica.ipynb`](../practicas/7_practica.ipynb).